# **🇹🇭 Constituency OCR: The Experimental AutoGluon Classification Approach (Shadow-Core)**

Unlike **Leaf Classification** or **Dog Breed Identification** which map one image to one label, Constituency OCR maps one 5-page form to 67 numbers. We cannot blindly feed the whole document to AutoGluon.

**The Experimental Strategy:**
We will bend the rules of reality by converting OCR into an Image Classification task!
1. **Synthetic Training Data**: Since we have no training digits, we generate synthetic digit images (0-9) and heavily augment them using `albumentations` (just like Dog/Leaf tasks).
2. **Train AutoGluon**: Train `autogluon.multimodal` on these synthetic digits.
3. **OpenCV Slicing**: Use OpenCV morphological operations to detect table grid lines, crop the exact cells containing votes.
4. **Inference**: Pass the cropped digit images to AutoGluon to classify them, and merge the predictions back to the CSV.

In [ ]:
# Install AutoGluon and Computer Vision libraries
!pip install -q autogluon albumentations opencv-python pandas pillow

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import albumentations as A
from autogluon.multimodal import MultiModalPredictor
import matplotlib.pyplot as plt
from glob import glob
import warnings
warnings.filterwarnings('ignore')

## 1. Synthesize Training Data & Augmentation
We generate 5,000 synthetic images of digits (0-9), applying rotation, blur, and noise.

In [ ]:
DATASET_DIR = './synthetic_digits'
IMG_DIR = os.path.join(DATASET_DIR, 'images')
os.makedirs(IMG_DIR, exist_ok=True)

# Albumentations pipeline
transform = A.Compose([
    A.Rotate(limit=15, p=0.8),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
    A.RandomBrightnessContrast(p=0.5),
])

train_data = []
samples_per_class = 500

print("Generating heavily augmented synthetic digits...")
for digit in range(10):
    for i in range(samples_per_class):
        # Create base image
        base_img = Image.new('L', (64, 64), color=255)
        draw = ImageDraw.Draw(base_img)
        
        # Random positioning to simulate jitter
        x, y = np.random.randint(10, 30), np.random.randint(5, 20)
        draw.text((x, y), str(digit), fill=0) # Need a ttf font for real robustness, using default here
        
        img_arr = np.array(base_img)
        augmented = transform(image=img_arr)['image']
        
        img_name = f"{digit}_{i}.jpg"
        img_path = os.path.join(IMG_DIR, img_name)
        
        cv2.imwrite(img_path, augmented)
        train_data.append({'image': img_path, 'label': digit})

train_df = pd.DataFrame(train_data)
train_df.to_csv(os.path.join(DATASET_DIR, 'train.csv'), index=False)
print(f"Generated {len(train_df)} training images.")
train_df.head()

## 2. Train AutoGluon MultiModalPredictor
Train an Image Classifier on the synthetic digits dataset.

In [ ]:
print("Training AutoGluon Image Classifier...")
predictor = MultiModalPredictor(
    label='label', 
    problem_type='multiclass',
    path='ag_digit_model'
)

# In a real run, increase time_limit. Here we do a fast prototype limit.
predictor.fit(train_df, time_limit=180)
print("Training Complete!")

## 3. OpenCV Table Grid Extraction & Inference
We use morphological operations to detect vertical and horizontal lines of the constituency forms to automatically crop vote cells.

In [ ]:
def slice_grid_and_predict(image_path):
    """
    Uses OpenCV to find table cells, assumes the right-most cells contain votes,
    crops them, and uses AutoGluon to predict the digit.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return {}
    
    # Binarize image
    _, thresh = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY_INV)
    
    # Detect horizontal lines
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    hor_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    
    # Detect vertical lines
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    ver_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    
    # Create grid mask
    table_mask = cv2.addWeighted(hor_lines, 0.5, ver_lines, 0.5, 0.0)
    _, table_mask = cv2.threshold(table_mask, 50, 255, cv2.THRESH_BINARY)
    
    # Find Contours (Bounding Boxes for cells)
    contours, _ = cv2.findContours(table_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    # Sort and filter contours (highly experimental layout logic)
    bounding_boxes = [cv2.boundingRect(c) for c in contours]
    bounding_boxes = [b for b in bounding_boxes if b[2] > 20 and b[3] > 20] # Filter small noise
    
    extracted_votes = []
    
    # Loop through detected boxes, extract crop, feed to AutoGluon
    # Note: In a real scenario, you map X,Y coords to Party Names. 
    # Here, we just demonstrate AutoGluon extraction logic.
    for x, y, w, h in bounding_boxes[:10]: # Just process first 10 for demo
        crop = img[y:y+h, x:x+w]
        
        crop_path = '/tmp/temp_crop.jpg'
        cv2.imwrite(crop_path, crop)
        
        # Use trained AutoGluon to predict digit!
        try:
            prediction = predictor.predict({'image': [crop_path]})
            pred_digit = str(prediction.iloc[0])
            extracted_votes.append(pred_digit)
        except:
            pass
            
    return extracted_votes

## 4. Full Pipeline Demonstration
Applying the Shadow-Core logic to the competition template.

In [ ]:
sub_df = pd.read_csv('./data/submission_template.csv')
doc_ids = sub_df['doc_id'].unique()

for doc_id in doc_ids[:2]: # Demo on 2 documents
    print(f"Processing Document: {doc_id} using OpenCV + AutoGluon")
    # Pass the first page of the document
    img_path = f"./data/images/{doc_id}.png"
    if os.path.exists(img_path):
        votes_found = slice_grid_and_predict(img_path)
        print(f"AutoGluon Detected Digits: {votes_found}")
        
# To run a real submission, map the extracted rows correctly to party names and write back to sub_df.